# 01 — Baseline Model

**Amaç:** MNIST üzerinde küçük ve kontrol edilebilir MLP baseline modelini kurmak ve başlangıç ölçümlerini kaydetmek.

Architecture: `784 → 128 → ReLU → 64 → ReLU → 10`. Seed, optimizer, learning rate, batch size, epoch, accuracy ve süre kayıt altına alınır.

In [ ]:
import os, sys, time, json
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.append(ROOT)
from src.model import build_model
from src.evaluation import accuracy
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
SEED = 42
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 5
transform = transforms.ToTensor()
train = datasets.MNIST('data', train=True, download=True, transform=transform)
test = datasets.MNIST('data', train=False, download=True, transform=transform)
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test, batch_size=BATCH_SIZE)
model = build_model(SEED).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()
start = time.time()
for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    print(f'epoch={epoch+1} test_accuracy={accuracy(model, test_loader, device):.4f}')
final_accuracy = accuracy(model, test_loader, device)
training_seconds = time.time() - start
os.makedirs('../results', exist_ok=True)
torch.save(model.state_dict(), '../results/baseline_model.pt')
config = {'seed': SEED, 'architecture': '784-128-ReLU-64-ReLU-10', 'optimizer': 'Adam', 'learning_rate': LR, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'accuracy': final_accuracy, 'training_seconds': training_seconds}
with open('../results/baseline_config.json', 'w', encoding='utf-8') as f: json.dump(config, f, indent=2)
print('saved baseline_model.pt')
print('final_accuracy:', final_accuracy)
print('training_seconds:', round(training_seconds, 2))

## Baseline kayıt
Gerçek accuracy, süre ve deney parametreleri `notes/experiment_log.md` dosyasına aktarılacaktır. Sonuçlar çalıştırmadan önce varsayılmamalıdır.